In [47]:
import numpy as np
import pandas as pd

In [66]:
bioinfo = pd.read_parquet('../data/raw/bioinfo.parquet')
ideology = pd.read_parquet('../data/raw/ideology.parquet')
terms = pd.read_parquet('../data/raw/terms.parquet')


In [ ]:

bioinfo.info()

In [49]:
bioinfo['bioguide_id'].duplicated().value_counts()

bioguide_id
False    555
Name: count, dtype: int64

In [50]:
bioinfo.head(3).T

,0,1,2
bioguide_id,R000575,S001185,P000609
Full name,Mike Rogers,Terri A. Sewell,Gary J. Palmer
Chamber,House of Representatives,House of Representatives,House of Representatives
State,Alabama,Alabama,Alabama
Party,Republican,Democratic,Republican
District,3.0,7.0,6.0
birthYear,1958,1965,1954
image,https://www.congress.gov/img/member/116_rp_al_...,https://www.congress.gov/img/member/s001185_20...,https://www.congress.gov/img/member/p000609_20...
Office address,"2469 Rayburn House Office Building, Washington...","1035 Longworth House Office Building, Washingt...","170 Cannon House Office Building, Washington, ..."
Phone,(202) 225-3261,(202) 225-2665,(202) 225-4921


In [51]:
def pandas_df_to_dbml(df: pd.DataFrame, table_name: str) -> str:
    """
    Converts a pandas DataFrame to a DBML string.

    Args:
        df: The pandas DataFrame to convert.
        table_name: The name of the table in the DBML schema.

    Returns:
        A DBML string representing the DataFrame schema.
    """

    dbml_string = f"Table {table_name} {{\n"

    for column_name, column_type in df.dtypes.items():
        dbml_type = map_pandas_dtype_to_dbml_type(column_type)
        dbml_string += f"  {column_name} {dbml_type}\n"

    dbml_string += "}\n"
    return dbml_string

def map_pandas_dtype_to_dbml_type(dtype) -> str:
    """Maps a pandas dtype to a DBML type."""
    dtype_name = str(dtype)
    if "int" in dtype_name:
      return "int"
    if "float" in dtype_name:
      return "float"
    if "datetime" in dtype_name:
        return "datetime"
    return "varchar"

In [52]:
bioinfo.columns = [x.replace(' ', '_').lower()  for x in bioinfo.columns]

print(pandas_df_to_dbml(bioinfo, 'members'))

Table members {
  bioguide_id varchar
  full_name varchar
  chamber varchar
  state varchar
  party varchar
  district float
  birthyear varchar
  image varchar
  office_address varchar
  phone varchar
  website varchar
}



In [53]:
ideology.info()

<class 'pandas.DataFrame'>
RangeIndex: 557 entries, 0 to 556
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   bioname              557 non-null    str    
 1   chamber              557 non-null    str    
 2   left_right_ideology  555 non-null    float64
 3   state_abbrev         557 non-null    str    
 4   district_code        557 non-null    int64  
 5   icpsr                557 non-null    int64  
 6   bioguide_id          556 non-null    str    
 7   party                557 non-null    str    
 8   fec_id               538 non-null    object 
dtypes: float64(1), int64(2), object(1), str(5)
memory usage: 60.7+ KB


In [54]:
ideology = ideology[['icpsr', 'bioguide_id', 'fec_id', 'left_right_ideology']]

In [55]:
fec = ideology.explode('fec_id')
ideology = ideology.drop(['fec_id'], axis=1)
fec = fec.drop(['left_right_ideology'], axis=1)

In [56]:
ideology

,icpsr,bioguide_id,left_right_ideology
0,99912,NaN,0.403
1,20301,R000575,0.379
2,21102,S001185,-0.402
3,21500,P000609,0.670
4,22140,M001212,0.646
...,...,...,...
552,42504,J000312,0.562
553,29940,B001230,-0.484
554,41111,J000293,0.648
555,20953,L000571,0.682


In [57]:
fec

,icpsr,bioguide_id,fec_id
0,99912,NaN,NaN
1,20301,R000575,H2AL03032
2,21102,S001185,H0AL07086
3,21500,P000609,H4AL06098
4,22140,M001212,H8AL02171
...,...,...,...
553,29940,B001230,S2WI00219
554,41111,J000293,S0WI00197
555,20953,L000571,H8WY00148
555,20953,L000571,S0WY00137


In [58]:
ideology = ideology.groupby(['bioguide_id']).agg({'left_right_ideology': 'mean'}).reset_index()

In [59]:
members = pd.merge(bioinfo, ideology,
                   on = 'bioguide_id',
                   how = 'outer',
                   validate = 'one_to_one',
                   indicator = 'matched')
members['matched'].value_counts()
members = members.drop(['matched'], axis=1)

In [60]:
ideology[ideology['bioguide_id'].duplicated()]

,bioguide_id,left_right_ideology


In [61]:
ideology.query("bioguide_id == 'K000401'")

,bioguide_id,left_right_ideology
277,K000401,0.2025


In [62]:
print(pandas_df_to_dbml(members, 'members'))

Table members {
  bioguide_id varchar
  full_name varchar
  chamber varchar
  state varchar
  party varchar
  district float
  birthyear varchar
  image varchar
  office_address varchar
  phone varchar
  website varchar
  left_right_ideology float
}



In [65]:
print(pandas_df_to_dbml(fec, 'crosswalk'))

Table crosswalk {
  icpsr int
  bioguide_id varchar
  fec_id varchar
}



In [67]:
terms.info()

<class 'pandas.DataFrame'>
RangeIndex: 3267 entries, 0 to 3266
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   bioguide_id  3267 non-null   str    
 1   chamber      3267 non-null   str    
 2   congress     3267 non-null   int64  
 3   stateCode    3267 non-null   str    
 4   startYear    3267 non-null   int64  
 5   endYear      3267 non-null   int64  
 6   district     2523 non-null   float64
dtypes: float64(1), int64(3), str(3)
memory usage: 272.4 KB


In [69]:
terms = terms.drop(['startYear', 'endYear'], axis=1)
terms.columns = [x.lower() for x in terms.columns]
print(pandas_df_to_dbml(terms, "terms"))

Table terms {
  bioguide_id varchar
  chamber varchar
  congress int
  statecode varchar
  district float
}



In [70]:
members.to_csv("../data/processed/members.csv", index=False)
fec.to_csv("../data/processed/crosswalk.csv", index=False)
terms.to_csv("../data/processed/terms.csv", index=False)